<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=350095850" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PPE YOLOv8 training (Colab / Kaggle)

Use **this notebook** for E0–E4. Local 8GB GPUs are only for baseline val, export, and `--batch 8` smokes.

Product bars: **vest / no_vest 95%+**, helmets next, goggles ~70% OK. **Boots are not in this cycle.**

1. Runtime → GPU (Colab) or GPU accelerator (Kaggle).
2. Add secret `ROBOFLOW_API_KEY` (Colab userdata / Kaggle Add-ons → Secrets).
3. Run all cells. Default experiment: `e0_n` on the 12k subset after Combined download + remap.

Prefer **one** of Colab or Kaggle, not both.

In [ ]:
import os
from pathlib import Path

# Colab secret, then Kaggle, then env (local fallback).
try:
    from google.colab import userdata
    os.environ.setdefault("ROBOFLOW_API_KEY", userdata.get("ROBOFLOW_API_KEY"))
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("ROBOFLOW_API_KEY", UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    except Exception:
        pass

assert os.environ.get("ROBOFLOW_API_KEY"), "Set ROBOFLOW_API_KEY as a Colab/Kaggle secret"
print("key_set", "colab" if IN_COLAB else "kaggle_or_local")

REPO = Path("/content/ppe") if IN_COLAB else Path("/kaggle/working/ppe")
if (REPO / "scripts" / "train.py").exists():
    # Repo already checked out from a previous run in this session (Kaggle/Colab
    # keep /kaggle/working and /content between cell reruns) — pull latest instead
    # of silently training against a stale, possibly-already-fixed-upstream copy.
    print(f"{REPO} already exists — pulling latest instead of re-cloning")
    !git -C {REPO} fetch --depth 1 origin main
    !git -C {REPO} reset --hard origin/main
else:
    REPO.mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git {REPO}
os.chdir(REPO)
print("cwd", Path.cwd())
!git -C {REPO} log -1 --oneline

In [2]:
%pip install -q ultralytics roboflow pyyaml opencv-python-headless
%pip install -q -e .
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ppe (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
cuda True Tesla T4


In [3]:
# Combined (~2.4GB zip) then Hard Hat Universe. Construction is optional for mapped eval.
!python scripts/download_datasets.py --execute --only combined hardhat
!python scripts/remap_labels.py --source data/raw/combined --out data/processed/combined --mapping combined
!python scripts/remap_labels.py --source data/raw/hardhat --out data/processed/hardhat --mapping hhu
!python scripts/make_subset.py --source data/processed/combined --out data/raw/combined_12k --n 12000 --seed 42
!python scripts/analyze_distribution.py

Requesting export roboflow-universe-projects/personal-protective-equipment-combined-model/4/yolov8 -> /kaggle/working/ppe/data/raw/combined
  zip 100.0% [2542363062/2542363062 bytes]
Extracting combined.zip -> /kaggle/working/ppe/data/raw/combined
Downloaded combined to /kaggle/working/ppe/data/raw/combined
Hard Hat Universe: using version 26 (prefer 26 no_nulls_plain)
Requesting export universe-datasets/hard-hat-universe-0dy7t/26/yolov8 -> /kaggle/working/ppe/data/raw/hardhat
  zip 100.0% [245677789/245677789 bytes]
Extracting hardhat.zip -> /kaggle/working/ppe/data/raw/hardhat
Downloaded hardhat to /kaggle/working/ppe/data/raw/hardhat
train: 30765 images, dropped 0 unmapped boxes
valid: 8814 images, dropped 0 unmapped boxes
test: 4423 images, dropped 0 unmapped boxes
Wrote remapped dataset to /kaggle/working/ppe/data/processed/combined
train: 4912 images, dropped 0 unmapped boxes
valid: 1414 images, dropped 0 unmapped boxes
test: 708 images, dropped 0 unmapped boxes
Wrote remapped da

In [ ]:
# E0 on 12k. Swap --exp: e1_s | e2_focal | e3_augs | e4_full44k
# P100/T4: batch 16. If OOM, add --batch 8.
#
# IMPORTANT: redirect to a log file instead of letting output stream into this
# cell. A 100-epoch run's carriage-return-updating progress bars, captured
# verbatim into the notebook's cell-output JSON, can bloat that JSON to the
# point where Kaggle's post-run nbconvert step (which always runs, converting
# the executed notebook to .ipynb/.html for the Output tab) takes HOURS to
# process it — the kernel looks "still running" and keeps billing the whole
# time, even though training itself finished long before. Confirmed exactly
# this happened on 2026-09-15/16: nbconvert's regex-based cell-output
# processing (mistune.py / filter_links.py) took ~2.8 hours on one bloated
# cell alone. Redirecting keeps this cell's own output tiny (just the tail)
# while the full log still lands on disk.
LOG = "/kaggle/working/train_e0_n.log"
!python scripts/train.py --exp e0_n --device 0 --batch 16 > {LOG} 2>&1
print(f"Full log: {LOG}")
!tail -n 60 {LOG}

In [ ]:
# Sanity-check the just-trained checkpoint on the held-out test split RIGHT NOW,
# while GPU/credits are still available — catches a systemic bug (e.g. a
# train/eval data mismatch) in the same session instead of hours or days
# later. This is exactly the check that caught configs/data/combined.yaml
# silently pointing training at unremapped raw labels (fixed 2026-09-16) —
# every class scored near-zero here when that bug was present, instead of the
# ~0.5-0.9 mAP50 a real, correctly-labeled run gets per class.
EVAL_LOG = "/kaggle/working/eval_e0_n.log"
!python scripts/eval.py --weights runs/train/e0_n/weights/best.pt --split test --out results/analysis/eval_e0_n.json > {EVAL_LOG} 2>&1
!cat {EVAL_LOG}

import json
with open("results/analysis/eval_e0_n.json") as f:
    m = json.load(f)["metrics"]
print(f"\nSanity check: mAP50={m['map50']:.3f} mAP50-95={m['map50_95']:.3f} P={m['precision']:.3f} R={m['recall']:.3f}")
assert m["map50"] > 0.3, (
    "mAP50 is suspiciously low (<0.3) — almost certainly a train/eval data "
    "mismatch (check configs/data/*.yaml `path:` actually points at "
    "data/processed/..., not data/raw/...) rather than a genuinely weak "
    "model. Do NOT copy this checkpoint off the VM or use it for anything "
    "until this is understood — investigate before spending more credits."
)
print("Sanity check passed — checkpoint is trained against correctly-labeled data.")

The sanity-check cell above already confirms mAP50 looks real before you leave the VM. If it passed: copy `runs/train/e0_n/weights/best.pt` off the VM (Drive / Kaggle output), then run `scripts/calibrate.py` locally and lead the report with **vest / no_vest**.